In [ ]:
import sys, os
from pathlib import Path
import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt

PROJECT_PATH = Path(r"B:\College\DL\handwriting_autocomplete_system\phase3_style_transfer")
os.chdir(PROJECT_PATH)
sys.path.insert(0, str(PROJECT_PATH))

from lib.utils import yaml2config
from lib.alphabet import strLabelConverter
from networks.BigGAN_networks import Generator
from networks.module import StyleEncoder, StyleBackbone

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CHECKPOINT = PROJECT_PATH / 'server_files' / 'epoch_70.pth'
PRETRAINED_WID = PROJECT_PATH / 'pretrained' / 'wid_iam_new.pth'
IMG_HEIGHT, CHAR_WIDTH = 64, 32
print(f"Device: {DEVICE}")

In [ ]:
# Load models
cfg = yaml2config(str(PROJECT_PATH / 'configs' / 'gan_iam.yml'))
generator = Generator(**cfg.GenModel).to(DEVICE)
style_backbone = StyleBackbone(**cfg.StyBackbone).to(DEVICE)
style_encoder = StyleEncoder(**cfg.EncModel).to(DEVICE)

# Load weights
ckpt = torch.load(CHECKPOINT, map_location=DEVICE)
generator.load_state_dict(ckpt['generator'])
style_encoder.load_state_dict(ckpt['style_encoder'])
if PRETRAINED_WID.exists():
    style_backbone.load_state_dict(torch.load(PRETRAINED_WID, map_location=DEVICE)['StyleBackbone'])

generator.eval(); style_encoder.eval(); style_backbone.eval()
label_converter = strLabelConverter('_'.join(cfg.dataset.split('_')[:2]))
print(f"Models loaded (epoch {ckpt.get('epoch', 'N/A')})")

In [ ]:
def preprocess(img_path):
    """Load and preprocess image for style extraction."""
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    h, w = img.shape
    new_w = max(int(w * IMG_HEIGHT / h), CHAR_WIDTH * 2)
    new_w = new_w + (CHAR_WIDTH - new_w % CHAR_WIDTH) % CHAR_WIDTH
    img = cv2.resize(img, (new_w, IMG_HEIGHT), interpolation=cv2.INTER_AREA if new_w < w else cv2.INTER_LINEAR)
    if img.mean() > 127: img = 255 - img  # Invert if light background
    tensor = torch.from_numpy((img / 255.0 - 0.5) / 0.5).float().unsqueeze(0).unsqueeze(0)
    return tensor, new_w

def to_image(tensor):
    """Convert tensor to displayable image."""
    img = tensor.squeeze().cpu().numpy()
    return np.clip((1 - (img + 1) / 2) * 255, 0, 255).astype(np.uint8)

@torch.no_grad()
def generate(style_img_path, texts):
    """Generate handwriting with given style and texts."""
    img_tensor, width = preprocess(style_img_path)
    img_tensor = img_tensor.to(DEVICE)
    style = style_encoder(img_tensor, torch.tensor([width]).to(DEVICE), style_backbone, vae_mode=False)
    
    results = []
    for text in texts:
        lbs, lb_lens = label_converter.encode([text])
        fake = generator(style, lbs.to(DEVICE), lb_lens.to(DEVICE))
        results.append((text, to_image(fake)))
    return results

In [ ]:
# === Generate ===
STYLE_IMAGE = PROJECT_PATH / 'inference' / 'input' / 'sample.png'  # Your style reference
TEXTS = ['hello', 'world', 'python', 'neural', 'network']

if STYLE_IMAGE.exists():
    results = generate(STYLE_IMAGE, TEXTS)
    
    fig, axes = plt.subplots(len(results) + 1, 1, figsize=(12, 2 * (len(results) + 1)))
    ref = cv2.imread(str(STYLE_IMAGE), cv2.IMREAD_GRAYSCALE)
    axes[0].imshow(ref, cmap='gray'); axes[0].set_title('Reference Style'); axes[0].axis('off')
    
    for i, (text, img) in enumerate(results):
        axes[i+1].imshow(img, cmap='gray'); axes[i+1].set_title(f'"{text}"'); axes[i+1].axis('off')
    
    plt.tight_layout(); plt.show()
else:
    print(f"Put a style reference image at: {STYLE_IMAGE}")

In [ ]:
# === Batch process all images in input folder ===
INPUT_DIR = PROJECT_PATH / 'inference' / 'input'
OUTPUT_DIR = PROJECT_PATH / 'inference' / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for img_path in INPUT_DIR.glob('*.png'):
    results = generate(img_path, TEXTS)
    
    fig, axes = plt.subplots(1, len(results), figsize=(3 * len(results), 2))
    for i, (text, img) in enumerate(results):
        axes[i].imshow(img, cmap='gray'); axes[i].set_title(text); axes[i].axis('off')
    
    plt.savefig(OUTPUT_DIR / f'{img_path.stem}_generated.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Processed: {img_path.name}")

print(f"Done! Results in {OUTPUT_DIR}")